# Empty-Car Hunting and Hollow Wear

Notebook 1 looked at the original files. Here I use the merged CSV to see whether differences between the left and right wheels of a wheelset are related to empty-car hunting. The merged data has one row per wheelset and rail profile. Hunting is recorded for the truck, which contains two wheelsets.

The data file goes in the project `data` folder. The chart and results in this notebook are for our internal analysis.

In [ ]:
# Shared input locations; see INSTALL.md for the Teams download layout.
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "project_paths.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the repository; see INSTALL.md.")
sys.path.insert(0, str(ROOT / "src"))
from project_paths import data_directory, find_project_root, merged_file
ROOT = find_project_root(ROOT)
DATA = data_directory(ROOT)

import matplotlib.pyplot as plt
import pandas as pd

project_folder = ROOT

data_folder = DATA
data_path = merged_file(DATA)

data = pd.read_csv(data_path)
print(f"Loaded {len(data)} wheelset/rail-profile rows from {data_path.name}.")

## Check the rows we need

For this analysis, each car/truck/rail-profile group should have two wheelsets and one empty-car hunting result. I check that before comparing the wear measurements.

In [ ]:
truck_key = ["Car", "Truck", "Rail_Profile"]
columns = truck_key + [
    "Axle", "Max Hollow Wear_L", "Max Hollow Wear_R", "Empty_Truck_Hunted"
]

if data[columns].isna().any().any():
    raise ValueError("One of the columns needed for this analysis has missing values.")

wheelset_counts = data.groupby(truck_key)["Axle"].nunique()
hunting_counts = data.groupby(truck_key)["Empty_Truck_Hunted"].nunique()

if (wheelset_counts != 2).any() or (hunting_counts != 1).any():
    raise ValueError("Check the wheelset counts or repeated hunting results.")

print(f"Truck/rail-profile assessments: {len(wheelset_counts)}")
print(f"Distinct trucks: {data[["Car", "Truck"]].drop_duplicates().shape[0]}")
print("Each assessment has two wheelsets and one hunting result.")

## Compare the wheels within each wheelset

I take the absolute difference between the left and right `Max Hollow Wear` values. For each truck/rail assessment, I use the larger difference of its two wheelsets. That gives one feature for each truck outcome while keeping the wheelset as the starting point.

In [ ]:
wheelsets = data[columns].copy()
wheelsets["left_right_difference"] = (
    wheelsets["Max Hollow Wear_L"] - wheelsets["Max Hollow Wear_R"]
).abs()

truck_results = (
    wheelsets.groupby(truck_key, as_index=False)
    .agg(
        largest_wheelset_difference=("left_right_difference", "max"),
        empty_car_hunted=("Empty_Truck_Hunted", "first"),
    )
)

summary = truck_results.groupby("empty_car_hunted")[
    "largest_wheelset_difference"
].agg(["count", "median"])
display(summary.rename(index={False: "No hunting recorded", True: "Hunting recorded"}))

## Look across rail profiles

Each truck appears under several rail profiles, so these are repeated assessments of the same trucks. The bars show medians for exploration; they are not independent samples or evidence of cause.

In [ ]:
median_by_rail = (
    truck_results.groupby(["Rail_Profile", "empty_car_hunted"])[
        "largest_wheelset_difference"
    ]
    .median()
    .unstack()
    .rename(columns={False: "No hunting recorded", True: "Hunting recorded"})
)

ax = median_by_rail.plot.bar(figsize=(9, 5), rot=0)
ax.set_title("Wheelset Hollow-Wear Difference and Empty-Car Hunting")
ax.set_xlabel("Rail profile")
ax.set_ylabel("Median largest wheelset difference (mm)")
ax.legend(title="Truck outcome")
plt.tight_layout()
plt.show()

## What I found

The merged file contains 392 axle/rail rows and 196 truck/rail assessments from
28 trucks. Each assessment has two wheelsets and a consistent acceleration
hunting label. The unworn reference is excluded by the merge, which explains
the difference from the original workbook's 203 assessments.

In this dataset, the largest left/right hollow-wear difference is generally
lower for hunting assessments, including within each rail profile in the plot.
This does not support the initial idea that a larger difference alone would
identify hunting. The plot is descriptive because the same trucks recur across
rails.

One reported hollow-wear value is much larger than the others. We kept the
source data unchanged; that measurement and its sign convention should be
checked before giving the size of the difference a physical interpretation.